In [46]:
import pandas as pd
import requests
import urllib.parse 
from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions


In [47]:
hrefs = []

for i in range(1,41):

    finca_raiz = r'https://www.fincaraiz.com.co'
    filter_apto_barraquilla = f'apartamentos/arriendo/barranquilla?pagina={i}'

    url_barranquilla = urllib.parse.urljoin(finca_raiz, filter_apto_barraquilla)

    driver = webdriver.Chrome()
    driver.get(url_barranquilla)

    print(f'Loading page: {i}')

    # cookies=driver.find_element(By.CSS_SELECTOR,'#__next > div.jss1')
    # cookies_btn = driver.find_element(By.CSS_SELECTOR,'#__next > div.jss1 > div > button')
    # if cookies:
    #     cookies_btn.click()

    html_body_next = driver.find_element(By.CSS_SELECTOR,'#__next').get_attribute('outerHTML')
    soup = BeautifulSoup(html_body_next,'html.parser')


    class_a_href = soup.find_all('a',href=True)
    for a_class in class_a_href:
        partial_url = a_class['href']
        url_size = len(partial_url)
        if url_size>60:
            full_url = urllib.parse.urljoin(base=finca_raiz, url=partial_url)
            hrefs.append(full_url)
    
    print(f'Current URL list length: {len(hrefs)}')

    driver.quit()



Loading page: 1
Current URL list length: 25
Loading page: 2
Current URL list length: 50
Loading page: 3
Current URL list length: 75
Loading page: 4
Current URL list length: 100
Loading page: 5
Current URL list length: 125
Loading page: 6
Current URL list length: 150
Loading page: 7
Current URL list length: 175
Loading page: 8
Current URL list length: 200
Loading page: 9
Current URL list length: 225
Loading page: 10
Current URL list length: 250
Loading page: 11
Current URL list length: 275
Loading page: 12
Current URL list length: 300
Loading page: 13
Current URL list length: 325
Loading page: 14
Current URL list length: 350
Loading page: 15
Current URL list length: 375
Loading page: 16
Current URL list length: 400
Loading page: 17
Current URL list length: 425
Loading page: 18
Current URL list length: 450
Loading page: 19
Current URL list length: 475
Loading page: 20
Current URL list length: 500
Loading page: 21
Current URL list length: 525
Loading page: 22
Current URL list length: 550


In [48]:
import requests
from bs4 import BeautifulSoup

# Inicializar el diccionario con listas vacías para cada clave
dicc = {
    "publicacion": [],
    'precio': [],
    'area_m2': [],
    'administracion': [],
    'habitaciones': [],
    'banos': [],
    'estrato': [],
    'parqueadero': [],
    'ascensor': [],
    'piscina': [],
    'antiguedad': [],
    'tipo_apartamento': [],
    'ubicacion_principal': [],
    'ubicacion_asociada': [],
    'descripcion_general': [],
    'link': []
}

for i, link in enumerate(hrefs):
    response = requests.get(link)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    class_p_content = soup.find_all('p')
    for j, line in enumerate(class_p_content):
        class_p_content[j] = str(line).split('>')[-2].split('<')[0]

    def get_value(need_word: str) -> any:
        try:
            idx_word = next(k for k, word in enumerate(class_p_content) if word == need_word)
            next_value = class_p_content[idx_word + 1]

            if need_word in ['Precio (COP)']:
                next_value = next_value.split()[-1].replace('.', '')
                return int(next_value)
            elif need_word in ['Administración']:
                next_value = next_value.split()[-2].replace('.', '')
                return int(next_value)
            elif need_word in ['Estrato', 'Baños', 'Parqueaderos', 'Habitaciones']:
                return int(next_value)
            elif need_word == 'Área construída':
                next_value = next_value.split()[0].replace('.', '')
                return int(next_value)
            elif need_word in ['Ascensor', 'Piscina']:
                return 'si'
            else:
                return str(next_value)
        except (StopIteration, IndexError, ValueError):
            return None

    def default_values(default_code: str) -> any:
        try:
            if default_code == "jss65 jss72 jss87":
                text = soup.find_all('p', class_=default_code)
                text = str(text).split()[-1].split('<')[0].replace('.','')
            else:
                text = soup.find_all('p', class_=default_code)
                text = str(text[0]).split('>')[1].split('<')[0]
            return text
        except IndexError:
            return None

    # Agregar valores al diccionario
    dicc['publicacion'].append(default_values('jss65 jss70 jss123'))
    dicc['precio'].append(default_values('jss65 jss72 jss87'))
    dicc['area_m2'].append(get_value('Área construída'))
    dicc['administracion'].append(get_value('Administración'))
    dicc['habitaciones'].append(get_value('Habitaciones'))
    dicc['banos'].append(get_value('Baños'))
    dicc['estrato'].append(get_value('Estrato'))
    dicc['parqueadero'].append(get_value('Parqueaderos'))
    dicc['ascensor'].append(get_value('Ascensor'))
    dicc['piscina'].append(get_value('Piscina'))
    dicc['antiguedad'].append(get_value('Antigüedad'))
    dicc['tipo_apartamento'].append(get_value('Tipo de apartamento'))
    dicc['ubicacion_principal'].append(get_value('Ubicación principal'))
    dicc['ubicacion_asociada'].append(get_value('Ubicaciones asociadas'))
    dicc['descripcion_general'].append(get_value('Descripción general'))
    dicc['link'].append(link)

    print(f'Páginas Cargadas: {i + 1}')

# Convertir el diccionario a un DataFrame
import pandas as pd

df = pd.DataFrame(dicc)


Páginas Cargadas: 1
Páginas Cargadas: 2
Páginas Cargadas: 3
Páginas Cargadas: 4
Páginas Cargadas: 5
Páginas Cargadas: 6
Páginas Cargadas: 7
Páginas Cargadas: 8
Páginas Cargadas: 9
Páginas Cargadas: 10
Páginas Cargadas: 11
Páginas Cargadas: 12
Páginas Cargadas: 13
Páginas Cargadas: 14
Páginas Cargadas: 15
Páginas Cargadas: 16
Páginas Cargadas: 17
Páginas Cargadas: 18
Páginas Cargadas: 19
Páginas Cargadas: 20
Páginas Cargadas: 21
Páginas Cargadas: 22
Páginas Cargadas: 23
Páginas Cargadas: 24
Páginas Cargadas: 25
Páginas Cargadas: 26
Páginas Cargadas: 27
Páginas Cargadas: 28
Páginas Cargadas: 29
Páginas Cargadas: 30
Páginas Cargadas: 31
Páginas Cargadas: 32
Páginas Cargadas: 33
Páginas Cargadas: 34
Páginas Cargadas: 35
Páginas Cargadas: 36
Páginas Cargadas: 37
Páginas Cargadas: 38
Páginas Cargadas: 39
Páginas Cargadas: 40
Páginas Cargadas: 41
Páginas Cargadas: 42
Páginas Cargadas: 43
Páginas Cargadas: 44
Páginas Cargadas: 45
Páginas Cargadas: 46
Páginas Cargadas: 47
Páginas Cargadas: 48
P

In [49]:
df.to_csv(path_or_buf='raw_data_arriendo.csv',index=False,encoding='utf-8')